# L15c: Graph Neural Networks
This lecture introduces _graph neural networks_ (GNNs), a family of neural networks built around the idea that the input is itself a graph. Unlike a multilayer perceptron, which operates on a flat feature vector, or a recurrent network, which operates on an ordered sequence, a GNN operates on a set of nodes connected by edges, with each node carrying a feature vector and each edge optionally carrying its own attributes.

The unit operation of a GNN is _message passing_: each node updates its own feature vector by aggregating the feature vectors of its neighbors and combining them with its own through a learned linear transform and a nonlinearity. Stacking $L$ such layers gives every node a representation that depends on its $L$-hop neighborhood in the graph. We focus on the symmetric-normalized graph convolution of [Kipf and Welling (2017)](https://arxiv.org/abs/1609.02907), state the message-passing framework that generalizes it, and survey the alternative aggregators GraphSAGE and GAT.

The body of the lecture follows the matrix-form derivation in [Wu et al. (2019), "A Comprehensive Survey on Graph Neural Networks"](https://arxiv.org/abs/1901.00596) and the introductory two-thirds of the [Stanford CS224W course](https://web.stanford.edu/class/cs224w/) (Lectures 3 and 4 of the 2024 offering, slides in `L15c/docs/`).

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Write down the GCN propagation rule and identify the role of each matrix:__ State the symmetric-normalized propagation matrix $\hat{\mathbf{D}}^{-1/2}(\mathbf{A} + \mathbf{I})\hat{\mathbf{D}}^{-1/2}$, the per-layer weight matrix, and the elementwise nonlinearity, and give the shape of each.
> * __Cast a GNN layer in the message-passing framework:__ Identify the message function, the permutation-invariant aggregator over the neighborhood, and the update function in GCN, GraphSAGE, and GAT, and explain why each is permutation invariant by construction.
> * __Match a GNN architecture to a learning task:__ Identify the readout used for node classification, graph classification, and link prediction, and state which loss is minimized at the graph level versus the node level.

Let's get started!
___

## Example
Today, we will use the following notebook to illustrate key concepts:

> [▶ Untrained GCN propagation on Zachary's Karate Club](CHEME-5820-L15c-Example-GraphNeuralNetworks-Spring-2026.ipynb). In this example, we build the symmetric-normalized graph convolution of [Kipf and Welling (2017)](https://arxiv.org/abs/1609.02907) by hand on Zachary's Karate Club (34 nodes, 78 edges, two communities). The weight matrices are random and never trained. We sweep the depth $L$ of the GCN stack and project the resulting node embeddings to two dimensions to show that the propagation rule already separates the two communities at $L = 2$, and that pushing $L$ much further produces _oversmoothing_ in which all node embeddings collapse toward each other.

___

## Why Graph Neural Networks, and What is a Graph?
The neural networks we have studied so far assume the input has a fixed regular structure. An MLP sees a length-$d$ vector. An RNN sees a sequence of length $T$, with a single ordering on the inputs and a recurrence on a hidden state. The architecture is tied to that structure: an MLP's first dense layer fixes the input dimension, and an RNN's recurrence assumes a single ordering on the inputs. Many problems do not have a regular structure. A molecule is a set of atoms connected by bonds. A social network is a set of users connected by friendships. A citation graph is a set of papers connected by references. In each case the data is naturally a _graph_, and an MLP or RNN that ignores the graph structure throws away the information that connects the entities.

A GNN has to meet two demands that the conventional architectures cannot: __variable input size and connectivity__, since the number of nodes and the connectivity pattern can change from one input to the next, and __permutation invariance over neighbors__, since the label of a node should not depend on the order in which we list its neighbors. The aggregation operator a GNN uses to combine the features of a node's neighbors must therefore be invariant to permutations of the neighbor set. The message-passing framework we develop next satisfies both of these by construction.

We work with an undirected graph $G = (\mathcal{V}, \mathcal{E})$ with $N = \lvert\mathcal{V}\rvert$ nodes and $\lvert\mathcal{E}\rvert$ edges. Each node $i\in\mathcal{V}$ carries a feature vector $\mathbf{x}_{i}\in\mathbb{R}^{d}$, and we collect these vectors as rows of a feature matrix $\mathbf{X}\in\mathbb{R}^{N\times d}$.

> __Graph notation__
>
> * __Adjacency matrix__ $\mathbf{A}\in\{0, 1\}^{N\times N}$ with $A_{ij} = 1$ if $(i, j)\in\mathcal{E}$ and $0$ otherwise. For an undirected graph $\mathbf{A} = \mathbf{A}^{\top}$.
> * __Degree matrix__ $\mathbf{D}\in\mathbb{Z}_{\geq 0}^{N\times N}$, diagonal, with $D_{ii} = \sum_{j} A_{ij}$ the degree of node $i$.
> * __Neighborhood__ $\mathcal{N}(i) = \{j\in\mathcal{V}\,\colon\,(i, j)\in\mathcal{E}\}$, the set of nodes adjacent to $i$. We write $\mathcal{N}(i)\cup\{i\}$ for the closed neighborhood that includes $i$ itself.
> * __Feature matrix__ $\mathbf{X}\in\mathbb{R}^{N\times d}$, where row $i$ is the feature vector $\mathbf{x}_{i}$ of node $i$.
> * __Self-loop adjacency__ $\hat{\mathbf{A}} = \mathbf{A} + \mathbf{I}_{N}$, with diagonal degree matrix $\hat{\mathbf{D}}$ defined by $\hat{D}_{ii} = 1 + \sum_{j} A_{ij}$.

The adjacency matrix is the only structural object that enters the GNN; the rest of the network is dense linear algebra. Edge features $\mathbf{e}_{ij}\in\mathbb{R}^{d_{e}}$ can be added when present (e.g. bond type in a molecule), but the architectures we cover today operate on $\mathbf{A}$ and $\mathbf{X}$ alone.

___

## Message Passing, GCN, and Two Generalizations
A graph neural network is defined by what each node does at each layer. The standard recipe has three steps applied at every node $i$, at every layer $\ell$.

> __Message-Passing Layer__
>
> Let $\mathbf{h}_{i}^{(\ell)}\in\mathbb{R}^{d_{\ell}}$ be the feature vector of node $i$ at layer $\ell$, with $\mathbf{h}_{i}^{(0)} = \mathbf{x}_{i}$. Each layer applies, for every node $i\in\mathcal{V}$:
> $$
\boxed{
\begin{align*}
\text{(message)} \quad     & \mathbf{m}_{ij}^{(\ell)} = \phi^{(\ell)}\!\bigl(\mathbf{h}_{i}^{(\ell)},\,\mathbf{h}_{j}^{(\ell)},\,\mathbf{e}_{ij}\bigr),\quad j\in\mathcal{N}(i) \\
\text{(aggregate)} \quad   & \mathbf{a}_{i}^{(\ell)} = \bigoplus_{j\in\mathcal{N}(i)}\,\mathbf{m}_{ij}^{(\ell)} \\
\text{(update)} \quad      & \mathbf{h}_{i}^{(\ell + 1)} = \psi^{(\ell)}\!\bigl(\mathbf{h}_{i}^{(\ell)},\,\mathbf{a}_{i}^{(\ell)}\bigr)
\end{align*}}
> $$
> where $\phi^{(\ell)}:\mathbb{R}^{d_{\ell}}\times\mathbb{R}^{d_{\ell}}\times\mathbb{R}^{d_{e}}\rightarrow\mathbb{R}^{d_{m}}$ is the message function (typically a learnable linear transform), $\bigoplus:(\mathbb{R}^{d_{m}})^{*}\rightarrow\mathbb{R}^{d_{m}}$ is a permutation-invariant aggregator over the neighbor set (sum, mean, max, or attention-weighted sum), and $\psi^{(\ell)}:\mathbb{R}^{d_{\ell}}\times\mathbb{R}^{d_{m}}\rightarrow\mathbb{R}^{d_{\ell + 1}}$ is the update function (typically a single linear layer plus a nonlinearity).

After $L$ layers, node $i$'s representation $\mathbf{h}_{i}^{(L)}$ depends on the features of every node within $L$ hops of $i$ in the graph; stacking is the only way information from distant parts of the graph reaches a node. Two structural properties of this framework matter throughout. First, __permutation invariance__: the aggregator $\bigoplus$ is required to be permutation invariant over the neighbor set, so sum, mean, max, and softmax-weighted sum all qualify but concatenation does not. This is what lets the same layer process a node with three neighbors and a node with thirty. Second, __weight sharing across nodes__: the functions $\phi^{(\ell)}$ and $\psi^{(\ell)}$ are shared across all nodes at layer $\ell$, so the same parameters define what one layer means everywhere in the graph regardless of size. Different GNN architectures differ only in the choice of $\phi$, $\bigoplus$, and $\psi$; we work through three canonical choices.

[Kipf and Welling (2017)](https://arxiv.org/abs/1609.02907) propose the simplest message-passing layer that still works well on standard benchmarks. The message is a linear projection of the neighbor's feature vector, the aggregator is a degree-normalized sum, and the update is just the aggregated message followed by an elementwise nonlinearity. Casting the per-node recipe in matrix form gives a single propagation rule that updates the entire feature matrix in one matrix multiply.

> __GCN propagation rule__
>
> Let $\hat{\mathbf{A}} = \mathbf{A} + \mathbf{I}_{N}$ be the adjacency matrix with self-loops, $\hat{\mathbf{D}}$ its diagonal degree matrix, and $\mathbf{P} = \hat{\mathbf{D}}^{-1/2}\,\hat{\mathbf{A}}\,\hat{\mathbf{D}}^{-1/2}\in\mathbb{R}^{N\times N}$ the symmetric-normalized propagation matrix. With per-layer weight matrix $\mathbf{W}^{(\ell)}\in\mathbb{R}^{d_{\ell}\times d_{\ell + 1}}$ and elementwise activation $\sigma:\mathbb{R}\rightarrow\mathbb{R}$, the GCN propagation rule is
> $$
\boxed{
\mathbf{H}^{(\ell + 1)} = \sigma\!\left(\mathbf{P}\,\mathbf{H}^{(\ell)}\,\mathbf{W}^{(\ell)}\right),\qquad \mathbf{H}^{(0)} = \mathbf{X}\in\mathbb{R}^{N\times d_{0}}
}
> $$
> where $\mathbf{H}^{(\ell)}\in\mathbb{R}^{N\times d_{\ell}}$ is the layer-$\ell$ feature matrix.

Three design choices are baked into this rule. The __self-loops__ $\mathbf{A} + \mathbf{I}$ fold a node's own previous feature into its update under the same weight matrix as its neighbors, so the message function is shared across self and neighbors. The __symmetric normalization__ $\hat{\mathbf{D}}^{-1/2}\,\hat{\mathbf{A}}\,\hat{\mathbf{D}}^{-1/2}$ replaces the entry $\hat{A}_{ij}$ with $1/\sqrt{\hat{d}_{i}\hat{d}_{j}}$, which keeps the largest eigenvalue of $\mathbf{P}$ at one and leaves layer activations on a stable scale; the matrix $\mathbf{P}$ is the same for every layer, only the weight matrix changes. The __single shared linear transform__ $\mathbf{W}^{(\ell)}$ multiplies every node's feature vector at layer $\ell$ in one dense matrix multiply, with cost per layer $\mathcal{O}(N\,d_{\ell}\,d_{\ell + 1} + \lvert\mathcal{E}\rvert\,d_{\ell})$.

> __Dimension dictionary (GCN layer)__
>
> * $\mathbf{X}\in\mathbb{R}^{N\times d_{0}}$ is the input feature matrix.
> * $\mathbf{H}^{(\ell)}\in\mathbb{R}^{N\times d_{\ell}}$ is the layer-$\ell$ feature matrix, with $\mathbf{H}^{(0)} = \mathbf{X}$.
> * $\mathbf{W}^{(\ell)}\in\mathbb{R}^{d_{\ell}\times d_{\ell + 1}}$ is the layer-$\ell$ learnable weight matrix.
> * $\mathbf{A}\in\{0, 1\}^{N\times N}$ is the adjacency matrix; $\hat{\mathbf{A}} = \mathbf{A} + \mathbf{I}_{N}$.
> * $\hat{\mathbf{D}}\in\mathbb{Z}_{\geq 0}^{N\times N}$ is the diagonal degree matrix of $\hat{\mathbf{A}}$.
> * $\mathbf{P} = \hat{\mathbf{D}}^{-1/2}\,\hat{\mathbf{A}}\,\hat{\mathbf{D}}^{-1/2}\in\mathbb{R}^{N\times N}$ is the propagation matrix, symmetric and fixed once $\mathbf{A}$ is fixed.
> * $\sigma:\mathbb{R}\rightarrow\mathbb{R}$ is the elementwise activation, applied componentwise. Common choices are ReLU and tanh.

Two layers of GCN suffice for many node-classification benchmarks. Stacking too deep collapses node embeddings toward each other, a phenomenon called __oversmoothing__: each application of $\mathbf{P}$ averages a node's features with those of its neighbors, so as $L$ grows every node's embedding approaches the same limit. The example notebook demonstrates this collapse explicitly by sweeping $L$ from $1$ to $20$ and tracking how the pairwise cosine similarity between node embeddings rises toward one.

GCN fixes the aggregator to a degree-normalized sum. Two prominent generalizations relax this choice and give the practitioner an explicit knob. [Hamilton, Ying, and Leskovec (2017)](https://arxiv.org/abs/1706.02216) propose __GraphSAGE__, which factors the GCN's "self plus neighbors" sum into two separate weight matrices and lets the user pick the aggregator. With learnable matrices $\mathbf{W}_{\text{self}}^{(\ell)}, \mathbf{W}_{\text{neigh}}^{(\ell)}\in\mathbb{R}^{d_{\ell}\times d_{\ell + 1}}$ and a permutation-invariant aggregator $\bigoplus$ (mean, max, or LSTM over a sampled subset of neighbors), the GraphSAGE update is
$$\mathbf{h}_{i}^{(\ell + 1)} = \sigma\!\left(\mathbf{W}_{\text{self}}^{(\ell)\top}\,\mathbf{h}_{i}^{(\ell)} + \mathbf{W}_{\text{neigh}}^{(\ell)\top}\,\Bigl(\bigoplus_{j\in\mathcal{N}(i)}\mathbf{h}_{j}^{(\ell)}\Bigr)\right).$$
The two-matrix split lets the network treat self-information and neighbor-information differently at the cost of doubling the parameter count per layer; the S2025 L15d lab's `MyCustomConvolutionLayerModel` uses precisely this split with a sum aggregator. [Veličković, Cucurull, Casanova, Romero, Liò, and Bengio (2018)](https://arxiv.org/abs/1710.10903) propose __GAT__, which replaces the fixed degree-based weighting of GCN with a _learned_ attention weighting over neighbors. With shared linear transform $\mathbf{W}^{(\ell)}\in\mathbb{R}^{d_{\ell}\times d_{\ell + 1}}$ and attention vector $\mathbf{a}^{(\ell)}\in\mathbb{R}^{2\,d_{\ell + 1}}$, the unnormalized attention score for the edge $(i, j)$ is $e_{ij}^{(\ell)} = \mathrm{LeakyReLU}\!\bigl(\mathbf{a}^{(\ell)\top}[\mathbf{W}^{(\ell)\top}\mathbf{h}_{i}^{(\ell)};\,\mathbf{W}^{(\ell)\top}\mathbf{h}_{j}^{(\ell)}]\bigr)$, normalized by softmax over each node's closed neighborhood to give $\alpha_{ij}^{(\ell)}$, and the GAT update is $\mathbf{h}_{i}^{(\ell + 1)} = \sigma(\sum_{j\in\mathcal{N}(i)\cup\{i\}}\alpha_{ij}^{(\ell)}\,\mathbf{W}^{(\ell)\top}\mathbf{h}_{j}^{(\ell)})$, typically applied with $K$ parallel attention heads whose outputs are concatenated. GAT replaces the fixed structural weighting $1/\sqrt{\hat{d}_{i}\hat{d}_{j}}$ with a content-aware weighting that depends on the features themselves.

> __GCN, GraphSAGE, and GAT in one table__
>
> | Layer | Message $\phi$ | Aggregator $\bigoplus$ | Update $\psi$ |
> |---|---|---|---|
> | GCN     | $\mathbf{W}^{(\ell)\top}\mathbf{h}_{j}^{(\ell)}$ | $\sum_{j\in\mathcal{N}(i)\cup\{i\}}1/\sqrt{\hat{d}_{i}\hat{d}_{j}}\,\bigl(\,\cdot\,\bigr)$ | $\sigma(\,\cdot\,)$ |
> | GraphSAGE | $\mathbf{h}_{j}^{(\ell)}$ | mean, max, or LSTM over $\mathcal{N}(i)$ | $\sigma\bigl(\mathbf{W}_{\text{self}}^{(\ell)\top}\mathbf{h}_{i}^{(\ell)} + \mathbf{W}_{\text{neigh}}^{(\ell)\top}\,\bigl(\,\cdot\,\bigr)\bigr)$ |
> | GAT     | $\alpha_{ij}^{(\ell)}\,\mathbf{W}^{(\ell)\top}\mathbf{h}_{j}^{(\ell)}$ | $\sum_{j\in\mathcal{N}(i)\cup\{i\}}\bigl(\,\cdot\,\bigr)$ | $\sigma(\,\cdot\,)$ |

___

## Tasks, Applications, and a Comparison to Other Architectures
A GNN computes a representation $\mathbf{H}^{(L)}\in\mathbb{R}^{N\times d_{L}}$ at the final layer. The choice of __readout__ on top of this representation determines what the network is predicting, and the rest of the architecture is unchanged across tasks.

> __Three GNN tasks__
>
> * __Node classification:__ Each node $i$ has a label $y_{i}\in\{1, \dots, C\}$ and the network predicts $\hat{y}_{i}$ from $\mathbf{h}_{i}^{(L)}$ directly, e.g. $\hat{\mathbf{y}}_{i} = \mathrm{softmax}\!\bigl(\mathbf{W}_{\text{out}}^{\top}\mathbf{h}_{i}^{(L)} + \mathbf{b}_{\text{out}}\bigr)$ with $\mathbf{W}_{\text{out}}\in\mathbb{R}^{d_{L}\times C}$. Training minimizes the cross-entropy over the labeled subset of nodes; the rest of the graph contributes its features and edges but not its labels (the _semi-supervised_ regime of the original GCN paper).
> * __Graph classification:__ Each graph $G$ has a single label $y_{G}\in\{1, \dots, C\}$. The per-node features are __pooled__ to a graph-level vector $\mathbf{h}_{G}\in\mathbb{R}^{d_{L}}$ via a permutation-invariant readout (mean, sum, or max over rows of $\mathbf{H}^{(L)}$), then $\mathbf{h}_{G}$ is passed to a dense classifier. This is the regime of the L15d lab on the MUTAG dataset.
> * __Link prediction:__ The network predicts whether an edge $(i, j)$ exists from the pair $(\mathbf{h}_{i}^{(L)}, \mathbf{h}_{j}^{(L)})$, typically via an inner-product or bilinear scorer $s_{ij} = \mathbf{h}_{i}^{(L)\top}\mathbf{M}\,\mathbf{h}_{j}^{(L)}$ followed by a sigmoid, with $\mathbf{M}\in\mathbb{R}^{d_{L}\times d_{L}}$. Training uses a binary cross-entropy over a sampled mix of true and corrupted edges.

The body of the network is identical across all three tasks; only the readout and the loss change. A growing literature applies graph-classification GNNs to molecular property prediction, where each molecule is a graph, atoms are nodes with chemical-element features, bonds are edges with bond-type features, and the graph-level label is a property such as toxicity, solubility, or binding affinity. [Zhang et al. (2022)](https://pubmed.ncbi.nlm.nih.gov/35074533/) survey GNN methods for drug-target interaction prediction, [Wang, Kumar, and Rajapakse (2025)](https://rdcu.be/ejGUL) report drug-mechanism prediction with explainable GNNs, and the L15d lab uses the same setup on the MUTAG dataset (mutagenic-effect prediction) as a small-scale instance of this regime.

A GNN sits in the same family of differentiable parameterized models as an MLP or RNN, but the assumed input structure and the unit operation are different on every axis that matters in practice.

> __Comparison of architectures__
>
> | Property | MLP | RNN | GNN |
> |---|---|---|---|
> | Input structure | Flat vector $\mathbb{R}^{d}$ | Sequence of length $T$ | Graph $(\mathcal{V}, \mathcal{E})$, variable $N$ |
> | Unit operation | Dense matmul | Recurrence on a hidden state | Aggregate over neighbors |
> | Weight sharing | None | Across time steps | Across nodes |
> | Permutation invariance | Not built in | Not built in | Built in (over neighbor sets) |
> | Receptive field after $L$ layers | Whole input | Last $L$ time steps | $L$-hop neighborhood |
> | Native data | Tabular | Sequences (text, audio) | Molecules, networks, citations |

Two consequences are worth highlighting. First, a GNN with $L$ layers can only see information that is at most $L$ hops away from a given node, so deep stacks are needed to propagate information across large graphs; in practice this is bounded by oversmoothing, which forces $L$ to stay small (usually $L\in\{2, 3, 4\}$). Second, the same architecture handles graphs of any size and shape without retraining, which is the property that makes the L15d lab able to train on $150$ MUTAG graphs of varying sizes and test on $38$ unseen ones with the same parameter set.

___

## Summary
A graph neural network is a stack of message-passing layers that aggregate information across the graph one hop per layer. The Kipf and Welling GCN specializes the message-passing framework by fixing the message function to a linear projection, the aggregator to a degree-normalized sum, and the update to "aggregated message followed by elementwise nonlinearity." The propagation matrix $\mathbf{P} = \hat{\mathbf{D}}^{-1/2}(\mathbf{A} + \mathbf{I})\hat{\mathbf{D}}^{-1/2}$ is a fixed function of the graph structure; only the per-layer weight matrices are learned. Different choices of message function and aggregator give GraphSAGE, GAT, and the broader family of message-passing networks.

> __Key Takeaways:__
>
> * __Message passing is the unit operation of a GNN:__ Each node updates its feature vector by computing a message from each neighbor, aggregating the messages with a permutation-invariant operator, and combining the aggregated message with its own feature vector through a learnable update. Stacking $L$ such layers gives every node a representation of its $L$-hop neighborhood.
> * __GCN is the simplest message-passing layer that works well in practice:__ The Kipf and Welling rule replaces the message function with a shared linear transform, the aggregator with a symmetric-normalized sum, and uses self-loops to fold a node's own features into its update. Only the per-layer weight matrices are learned; the propagation matrix is fixed by the graph.
> * __The same body supports node, graph, and link prediction:__ A node-classification readout reads each row of the final feature matrix; a graph-classification readout pools the rows to a single graph-level vector; a link-prediction readout scores pairs of rows. Only the readout and the loss change with the task.

The companion example walks through the GCN propagation rule on Zachary's Karate Club with random untrained weights to show that the symmetric-normalized propagation pulls nodes within the same community together at modest depth, and that increasing the depth eventually collapses every node embedding to roughly the same vector. The L15d lab takes the next step by training a stack of message-passing layers on a graph-classification dataset (MUTAG molecules) with a real loss and a real optimizer.
___